# LangChain Tutorial
## Semantic search in documents using Vector Store Retriever
### Import Libraries

In [ ]:
import json
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown, JSON, display_json

### Create a few dumy documents

In [9]:
documents = [
    Document(
        page_content="AcmeOrbit IT policy: Employees must return a borrowed laptop within 14 calendar days after the loan ends.", 
        metadata={"source": "IT-LAPTOP-01"}),
    Document(
        page_content="AcmeOrbit travel policy: Expense claims must be submitted within 30 calendar days after travel.", 
        metadata={"source": "TRAVEL-02"}),
    Document(
        page_content="AcmeOrbit security policy: Employees must use multifactor authentication for company accounts.", 
        metadata={"source": "SECURITY-03"}),
]


### Load from PDF

In [67]:
file_path = "../data/llama_research_paper.pdf"
loader = PyPDFLoader(file_path=file_path)

docs = loader.load()
print(len(docs))  # The number of pages on the document

27


### Check the content of one document

In [22]:
n = 0
print("# Check one Document")
print("Content:\n", docs[n].page_content[:200])
print("-"*60)
print("Metadata:\n")
metadata = docs[n].metadata
for key in metadata:
    print(f"* {key}: {metadata[key]}")

# Check one Document
Content:
 LLaMA: Open and Efﬁcient Foundation Language Models
Hugo Touvron∗, Thibaut Lavril∗, Gautier Izacard∗, Xavier Martinet
Marie-Anne Lachaux, Timothee Lacroix, Baptiste Rozière, Naman Goyal
Eric Hambro, F
------------------------------------------------------------
Metadata:

* producer: pdfTeX-1.40.21
* creator: LaTeX with hyperref
* creationdate: 2023-02-28T01:57:46+00:00
* author: 
* keywords: 
* moddate: 2023-02-28T01:57:46+00:00
* ptex.fullbanner: This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2
* subject: 
* title: 
* trapped: /False
* source: llama_research_paper.pdf
* total_pages: 27
* page: 0
* page_label: 1


In [33]:
print(json.dumps(dict(docs[0]), indent=2))

{
  "id": null,
  "metadata": {
    "producer": "pdfTeX-1.40.21",
    "creator": "LaTeX with hyperref",
    "creationdate": "2023-02-28T01:57:46+00:00",
    "author": "",
    "keywords": "",
    "moddate": "2023-02-28T01:57:46+00:00",
    "ptex.fullbanner": "This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2",
    "subject": "LLaMA: Open and Efficient Foundation Language Models",
    "title": "research paper Llama",
    "trapped": "/False",
    "source": "Meta AI",
    "total_pages": 27,
    "page": 0,
    "page_label": "1",
    "description": "\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best \n    large lang

### Update or Add Metadatas

In [23]:
description = """
    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. 
    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. 
    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best 
    large language models while only using publicly available data. The models and training methodology aim to improve access to large 
    language models and allow for open-sourcing of the models.
    """
for n in range(len(docs)):
    docs[n].metadata["title"] = "research paper Llama"
    docs[n].metadata["subject"] = "LLaMA: Open and Efficient Foundation Language Models"
    docs[n].metadata["description"] = description
    docs[n].metadata["source"] = "Meta AI"
    docs[n].metadata["type"] = "PDF"

### Split each documents intoi small chunck

In [32]:
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)
all_splits = text_spliter.split_documents(docs)

print(len(all_splits))
print(json.dumps(dict(all_splits[0]), indent=2))

118
{
  "id": null,
  "metadata": {
    "producer": "pdfTeX-1.40.21",
    "creator": "LaTeX with hyperref",
    "creationdate": "2023-02-28T01:57:46+00:00",
    "author": "",
    "keywords": "",
    "moddate": "2023-02-28T01:57:46+00:00",
    "ptex.fullbanner": "This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2",
    "subject": "LLaMA: Open and Efficient Foundation Language Models",
    "title": "research paper Llama",
    "trapped": "/False",
    "source": "Meta AI",
    "total_pages": 27,
    "page": 0,
    "page_label": "1",
    "description": "\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best \n    large 

### Creating and Store Embeddings 
#### Download HuggingFace models Once and Save/Load from a Local Directory
f you want to explicitly save the model files to a specific local folder (e.g., ./local_bge_small) and load directly from that path offline, download and save it via sentence-transformers or transformers.

In [37]:
local_model_path = "../models/local_bge_small"

# 1. Download and save to disk once
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
model.save(local_model_path)

# 2. Load HuggingFaceEmbeddings pointing to the local folder
embeddings = HuggingFaceEmbeddings(model_name=local_model_path)
vector_store = Chroma(embedding_function=embeddings)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Add the documents to the Vector Store

In [41]:
ids = vector_store.add_documents(documents=all_splits)
len(ids), ids[:5]

(118,
 ['a6a8a0ca-826e-4574-813e-1ac7258063fe',
  '61525eb4-640f-4524-bdf4-8a61591cb695',
  '13e985d5-0c50-4072-ab68-ddc0b1144acf',
  '1b2dc058-87a4-48cd-8f94-d2913c8d540b',
  '4e3380a1-776d-40dc-a588-0ab5696f9d10'])

### Perform semantic Search

In [45]:
query = "What is the optimizer used?"
results = vector_store.similarity_search(query=query)
print("Total chunks:" ,len(results))
print("First Chunk:")
print("-"*60)
print(json.dumps(dict(results[0]), indent=2))


Total chunks: 4
First Chunk:
------------------------------------------------------------
{
  "id": "e27e63bb-7ea6-4c7a-9df8-72f6f45c86cf",
  "metadata": {
    "ptex.fullbanner": "This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2",
    "moddate": "2023-02-28T01:57:46+00:00",
    "type": "PDF",
    "source": "Meta AI",
    "subject": "LLaMA: Open and Efficient Foundation Language Models",
    "start_index": 1567,
    "keywords": "",
    "title": "research paper Llama",
    "description": "\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best \n    large language models while only using publicly available data. The

In [46]:
print(results[0].page_content[:250])

The details of the hyper-parameters for our dif-
ferent models are given in Table 2.
2.3 Optimizer
Our models are trained using the AdamW opti-
mizer (Loshchilov and Hutter, 2017), with the fol-
lowing hyper-parameters: β1 = 0.9,β 2 = 0.95.
We use a 


#### similarity_search_with_relevance_scores - Score based on Logic Similarity
This method abstracts away the mathematical calculations and focuses on the end-user experience. It assumes that you want the most relevant documents first, ordering the results in descending order of logical relevance (with the best match at the top).

In [52]:
results = vector_store.similarity_search_with_relevance_scores(query)
print("Total chunks:" ,len(results))
print("First Chunk:")
print("-"*60)
print("Score: ", results[0][1])
print(json.dumps(dict(results[0][0]), indent=2))

Total chunks: 4
First Chunk:
------------------------------------------------------------
Score:  0.5853665536077683
{
  "id": "e27e63bb-7ea6-4c7a-9df8-72f6f45c86cf",
  "metadata": {
    "type": "PDF",
    "start_index": 1567,
    "creationdate": "2023-02-28T01:57:46+00:00",
    "creator": "LaTeX with hyperref",
    "title": "research paper Llama",
    "producer": "pdfTeX-1.40.21",
    "ptex.fullbanner": "This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2",
    "description": "\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best \n    large language models while only using publicly available data. The models and 

In [54]:
print("Scores by Chunks:")
for num, chunk in enumerate(results, start=1):
    print(f"Score of Chunk {num}: ", chunk[1])


Scores by Chunks:
Score of Chunk 1:  0.5853665536077683
Score of Chunk 2:  0.5600301442128386
Score of Chunk 3:  0.5539636954255744
Score of Chunk 4:  0.5117480211110621


#### similarity_search_by_vector_with_relevance_scores - Score based on Distance
This method exposes the raw data calculated by the underlying vector database (such as Chroma, FAISS, Pinecone, etc.). If your vector database is configured to use L2 (Euclidean) distance or Cosine distance (rather than Cosine similarity), a lower number indicates that the vectors are closer together in space. Therefore, the lowest score represents the greatest proximity (the best result). It lists the results from the smallest error/distance to the largest.

In [57]:
embedding = embeddings.embed_query(query)
results2 = vector_store.similarity_search_by_vector_with_relevance_scores(embedding=embedding)

print("Scores by Chunks:")
for num, chunk in enumerate(results, start=1):
    print(f"Score of Chunk {num}: ", chunk[1])

Scores by Chunks:
Score of Chunk 1:  0.5863802433013916
Score of Chunk 2:  0.6222113370895386
Score of Chunk 3:  0.6307905912399292
Score of Chunk 4:  0.690492570400238


In [58]:
results[0]

(Document(id='e27e63bb-7ea6-4c7a-9df8-72f6f45c86cf', metadata={'start_index': 1567, 'creator': 'LaTeX with hyperref', 'title': 'research paper Llama', 'total_pages': 27, 'description': '\n    This document introduces LLaMA, a collection of foundation language models ranging from 7B to 65B parameters trained by Meta AI. \n    The models are trained on trillions of tokens from publicly available datasets, achieving state-of-the-art performance. \n    Specifically, LLaMA-13B outperforms GPT-3 on most benchmarks despite being 10x smaller, and LLaMA-65B is competitive with the best \n    large language models while only using publicly available data. The models and training methodology aim to improve access to large \n    language models and allow for open-sourcing of the models.\n    ', 'author': '', 'page': 2, 'creationdate': '2023-02-28T01:57:46+00:00', 'source': 'Meta AI', 'page_label': '3', 'moddate': '2023-02-28T01:57:46+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX,

### Set a retriever to configure the serach and retriever process, like to get top n similar searchs

In [63]:
retriever = vector_store.as_retriever(search_type="similarity_score_threshold", search_kwargs={ "k": 3, "score_threshold": 0.5})
results = retriever.invoke(query)
print(len(results))

3


#### If you need this inside a Chain (LCEL)
If you are using this in a chain and absolutely need the scores passed to the next step, you can create a simple custom function:

In [66]:
def retrieve_with_scores(query):
    # Call the vector store directly inside the function
    return vector_store.similarity_search_with_relevance_scores(query, score_threshold=0.5, k=3)

def print_scores(results):
    print("Scores by Chunks:")
    for num, chunk in enumerate(results, start=1):
        print(f"Score of Chunk {num}: ", chunk[1])

# Wrap it in a RunnableLambda to use it in a chain
chain = RunnableLambda(retrieve_with_scores) | print_scores
chain.invoke(query)

Scores by Chunks:
Score of Chunk 1:  0.5853665536077683
Score of Chunk 2:  0.5600301442128386
Score of Chunk 3:  0.5539636954255744
